<a href="https://colab.research.google.com/github/gitesei/WSL_Simulation_Lab/blob/main/md_simulation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Water Simulation Lab
This Colab notebook enables running molecular dynamics (MD) simulations of pure water and studying structure and dynamics.

MD simulations employ the TIP4P2005 water model.

Simulations are run using OpenMM [4] at the user-defined temperature.

### Usage
We recommend running MD simulations run on a single GPU. To enable GPU select `Runtime` from the menu, then `Change runtime type` and select `GPU`.

Note: Cells for preliminary operations should be executed one by one to prevent crashes.

### References

4. P. Eastman, J. Swails, J. D. Chodera et al. __OpenMM 7: Rapid development of high performance algorithms for molecular dynamics__ _PLoS Comput Biol._ 2017 13(7):e1005659 DOI: https://doi.org/10.1371/journal.pcbi.1005659

In [ ]:
# @title 1. Set the environment for simulations and analyses
!pip install openmm[cuda12]
!pip install --upgrade MDAnalysis
!pip install mdtraj
!pip install -q py3Dmol gdown mrcfile &> /dev/null

In [ ]:
# @title 2. Configure your water system

import numpy as np
import os
import shutil
import ipywidgets as widgets
from IPython.display import display,Markdown
import warnings
import yaml
import mdtraj as md
import py3Dmol

warnings.filterwarnings('ignore')

temperature = 293.15 #@param {type:"number"}
box_side_length = 2.20 #@param {type:"number"}
simulation_time = 2 #@param {type:"number"}
#@markdown <i>*Units: temperature [K], box side length [nm], simulation time [ns]<i>

system_name = f"{box_side_length:.2f}_{temperature:.2f}"

if not os.path.isdir(f"{system_name}"):
    os.system(f"mkdir -p {system_name}")
    os.system(f"mkdir -p {system_name}/figures")

N_steps = simulation_time * 1000 / 0.002

config_sim_data = dict(temperature=float(temperature), box_side_length=box_side_length, N_steps=N_steps)
yaml.dump(config_sim_data, open(f'{system_name}/config_sim.yaml', 'w'))

In [ ]:
#@title <b><font color='#A79AB2'>MD simulation toolbox</font></b>

import time
from fastprogress import progress_bar
from openmm import *
from openmm.app import *
from openmm.unit import *

def run_steps(simulation, steps, chunk=1000):
    starttime = time.time()
    nblocks, remainder = divmod(steps, chunk)

    for _ in progress_bar(range(nblocks)):
        simulation.step(chunk)

    if remainder:
        simulation.step(remainder)

    elapsed = time.time()-starttime
    print(f"Simulation time: {elapsed//3600:.0f}h {(elapsed//60)%60:.0f}min {elapsed%60:.2f}s")

def simulate(config):
    temperature = config['temperature']
    box_side_length = config['box_side_length']
    N_steps = int(config['N_steps'])

    top = Topology()
    modeller = Modeller(top, [])

    ff = ForceField('charmm36.xml','charmm36/tip4p2005.xml')
    modeller.addSolvent(ff,model='tip4pew',boxSize=Vec3(box_side_length,box_side_length,box_side_length)*nanometer)

    n_atoms = modeller.topology.getNumAtoms()
    n_residues = modeller.topology.getNumResidues()
    n_waters = sum([res.name in {'HOH','WAT','TIP4'} for res in modeller.topology.residues()])
    n_ions = n_residues-n_waters

    print(f"Number of atoms: {n_atoms}")
    print(f"Number of water molecules: {n_waters}")

    dt = 0.002*picoseconds
    Temperature = temperature*kelvin
    integrator = LangevinMiddleIntegrator(Temperature,1/picosecond,dt)

    system = ff.createSystem(modeller.topology,nonbondedMethod=PME,nonbondedCutoff=1*nanometer,constraints=HBonds)
    simulation = Simulation(modeller.topology,system,integrator)
    simulation.context.setPositions(modeller.positions)

    state = simulation.context.getState(getEnergy=True)
    e_0 = state.getPotentialEnergy()
    print(f"Initial potential energy: {e_0}")

    simulation.minimizeEnergy()

    state = simulation.context.getState(getEnergy=True,getPositions=True)
    e_1 = state.getPotentialEnergy()
    print(f"Potential energy after minimization: {e_1}")
    print(f"Energy change: {e_1-e_0}")

    with open(f'{system_name}/EM_top.pdb','w') as f:
        PDBFile.writeFile(simulation.topology,state.getPositions(),f)

    simulation.context.setVelocitiesToTemperature(Temperature)

    simulation.reporters.append(StateDataReporter(f'{system_name}/NVT_log.txt',1000,step=True,temperature=True,potentialEnergy=True,density=True,volume=True))
    print("Run NVT for 50000 steps")
    run_steps(simulation,50000,1000)
    simulation.reporters.pop()

    system.addForce(MonteCarloBarostat(1*bar,Temperature,25))
    simulation.context.reinitialize(preserveState=True)

    simulation.reporters.append(StateDataReporter(f'{system_name}/NPT_log.txt',1000,step=True,temperature=True,potentialEnergy=True,density=True,volume=True))
    simulation.reporters.append(DCDReporter(f'{system_name}/NPT_traj.dcd',5000))

    print(f"Run NPT for {N_steps:d} steps")
    run_steps(simulation,N_steps,1000)

In [ ]:
# @title 3. Run MD simulation
config = yaml.safe_load(open(f'{system_name}/config_sim.yaml', 'r'))
simulate(config)

In [ ]:
# @title 4. Visualize the trajectory

traj = md.load_dcd("NPT_traj.dcd", top="EM_top.pdb")

def atom_line(serial, name, resname, resid, x, y, z, element):
    return f"HETATM{serial:5d} {name:<4s} {resname:>3s} A{resid:4d}    {x:8.3f}{y:8.3f}{z:8.3f}  1.00  0.00          {element:>2s}\n"

def conect_line(i, bonded):
    return f"CONECT{i:5d}" + "".join(f"{j:5d}" for j in bonded) + "\n"

def wrap_centered(x, box):
    return x - box*np.floor(x/box + 0.5)

def center_molecules(frame_xyz, top, box_lengths):
    xyz = frame_xyz.copy()

    for residue in top.residues:
        atoms = list(residue.atoms)
        inds = np.array([atom.index for atom in atoms])

        O = None
        for atom in atoms:
            if atom.name == "O":
                O = atom
                break
        if O is None:
            O = atoms[0]

        shift = box_lengths*np.floor(xyz[O.index]/box_lengths + 0.5)
        xyz[inds] -= shift

    return xyz

def frame_to_pdb_string(frame_xyz, top, box_lengths):
    lines = []
    serial = 1

    xyz = center_molecules(frame_xyz, top, box_lengths) * 10.0
    a, b, c = box_lengths * 10.0

    for atom, coord in zip(top.atoms, xyz):
        name = atom.name
        resname = atom.residue.name
        resid = atom.residue.index + 1
        element = atom.element.symbol if atom.element is not None else name[0]
        lines.append(atom_line(serial, name, resname, resid, coord[0], coord[1], coord[2], element))
        serial += 1

    x0, x1 = -a/2, a/2
    y0, y1 = -b/2, b/2
    z0, z1 = -c/2, c/2

    corners = [
        (x0,y0,z0), (x1,y0,z0), (x0,y1,z0), (x0,y0,z1),
        (x1,y1,z0), (x1,y0,z1), (x0,y1,z1), (x1,y1,z1)
    ]

    box_serials = []
    for i, (x, y, z) in enumerate(corners):
        lines.append(atom_line(serial, f"B{i+1}", "BOX", 999, x, y, z, "C"))
        box_serials.append(serial)
        serial += 1

    edges = [
        (0,1), (0,2), (0,3),
        (1,4), (1,5),
        (2,4), (2,6),
        (3,5), (3,6),
        (4,7), (5,7), (6,7)
    ]

    adjacency = {s: [] for s in box_serials}
    for i, j in edges:
        si, sj = box_serials[i], box_serials[j]
        adjacency[si].append(sj)
        adjacency[sj].append(si)

    for s in box_serials:
        lines.append(conect_line(s, adjacency[s]))

    return "".join(lines)

pdb_models = []
for i in range(traj.n_frames):
    pdb_models.append(f"MODEL     {i+1}\n")
    pdb_models.append(frame_to_pdb_string(traj.xyz[i], traj.topology, traj.unitcell_lengths[i]))
    pdb_models.append("ENDMDL\n")

pdb_string = "".join(pdb_models)

view = py3Dmol.view(width=650, height=500)
view.addModelsAsFrames(pdb_string, "pdb", {"keepH": True})
view.setBackgroundColor("white")

view.setStyle({}, {})
view.setStyle({'resn':'HOH', 'atom':'O'},  {'sphere':{'color':'red',   'radius':1.52}})
view.setStyle({'resn':'HOH', 'atom':'H1'}, {'sphere':{'color':'white', 'radius':1.20}})
view.setStyle({'resn':'HOH', 'atom':'H2'}, {'sphere':{'color':'white', 'radius':1.20}})
view.setStyle({'resn':'HOH', 'atom':'M'}, {})
view.setStyle({'resn':'BOX'}, {'stick':{'color':'black', 'radius':0.05}})

view.zoomTo()
view.animate({'loop':'forward', 'reps':1, 'interval':200})
view.show()

In [ ]:
#@title <b><font color='#A79AB2'>Analysis toolbox</font></b>
import matplotlib.pyplot as plt
import MDAnalysis as mda
import MDAnalysis.analysis.rdf as RDF
from MDAnalysis.analysis.hydrogenbonds.hbond_analysis import HydrogenBondAnalysis as HBA

def load_state_data(filename,columns):
    data = np.loadtxt(filename,delimiter=',')
    return {name:data[:,i] for i,name in enumerate(columns)}

def plot_timeseries(x,y,ylabel,title,average=False):
    plt.figure(figsize=(6,4))
    plt.plot(x*2e-6,y)

    if average:
        av = np.mean(y)
        print(f"Average {ylabel}: {av:.3f}")
        plt.axhline(av,color='tab:red',ls='--')

    plt.xlabel("Time (ns)")
    plt.ylabel(ylabel)
    plt.title(title)
    plt.tight_layout()
    plt.savefig(f'{system_name}/figures/{title.replace(' ','_')}.jpg',dpi=600)
    plt.show()

def add_water_bonds(u,oxygen='O',hydrogens=('H1','H2')):
    bonds = []

    for res in u.residues:
        O = res.atoms.select_atoms(f"name {oxygen}")

        for hydrogen in hydrogens:
            H = res.atoms.select_atoms(f"name {hydrogen}")

            if len(O) == 1 and len(H) == 1:
                bonds.append((O[0].index,H[0].index))

    u.add_bonds(bonds)

In [ ]:
# @title 5. Load the trajectory
u = mda.Universe(f"{system_name}/EM_top.pdb",f"{system_name}/NPT_traj.dcd")

print(f"Frames: {len(u.trajectory)}")
print(f"Atoms: {u.atoms.n_atoms}")
print(f"Water molecules: {len(u.select_atoms('resname HOH and name O'))}")

In [ ]:
# @title 5. NVT analysis
#@markdown In the NVT ensemble, the number of particles and volume are fixed while the temperature is controlled by a thermostat.
#@markdown
#@markdown **Questions**
#@markdown
#@markdown 1. Does the temperature fluctuate around the target temperature?
#@markdown 2. Is there a systematic increase or decrease in temperature?
#@markdown 3. Why is the density constant during an NVT simulation?

nvt = load_state_data(f"{system_name}/NVT_log.txt",['step','potential_energy','temperature','volume','density'])

plot_timeseries(nvt['step'],nvt['temperature'],"Temperature (K)","NVT temperature")
plot_timeseries(nvt['step'],nvt['density'],"Density (g/L)","NVT density")

In [ ]:
# @title 6. NPT analysis

#@markdown In the NPT ensemble, pressure and temperature are controlled while the box volume is allowed to fluctuate. Since the number of particles is fixed, fluctuations in volume result in fluctuations in density.
#@markdown
#@markdown **Questions**
#@markdown
#@markdown 1. Does the density fluctuate around a stable average, or does it show a clear trend?
#@markdown 2. Does the volume reach a stationary regime?
#@markdown 3. Are the volume and density fluctuations correlated or anticorrelated?
#@markdown 4. How close is the average density to the expected density of liquid water?
#@markdown 5. How many frames in the first part of the trajectory should be discarded as equilibration?

npt = load_state_data(f"{system_name}/NPT_log.txt",['step','potential_energy','temperature','volume','density'])

plot_timeseries(npt['step'],npt['temperature'],"Temperature (K)","NPT temperature",average=True)
plot_timeseries(npt['step'],npt['volume'],"Volume (nm$^3$)","NPT volume",average=True)
plot_timeseries(npt['step'],npt['density'],"Density (g/L)","NPT density",average=True)

In [ ]:
# @title 7. Hydrogen-bond analysis

#@markdown A hydrogen bond is identified using geometric criteria.

#@markdown For two water molecules:

#@markdown - the donor is the oxygen covalently bonded to a hydrogen;
#@markdown - the hydrogen is H1 or H2 on the donor molecule;
#@markdown - the acceptor is the oxygen on another water molecule.

#@markdown A hydrogen bond is counted when:

#@markdown 1. the donor–acceptor distance is smaller than 3.5 Å;
#@markdown 2. the donor–hydrogen–acceptor angle is larger than 140°.

#@markdown The distance criterion ensures that the two water molecules are sufficiently close. The angular criterion selects configurations in which the O–H bond points toward the acceptor oxygen.
#@markdown
#@markdown **Questions**
#@markdown
#@markdown 1. Why is a distance criterion alone insufficient?
#@markdown 2. What happens to the number of detected hydrogen bonds if the angle cutoff is reduced?
#@markdown 3. What happens if the distance cutoff is increased?
#@markdown 4. Why do hydrogen-bond counts fluctuate with time?

d_a_cutoff = 3.5 #@param {type:"number"}
d_h_a_angle_cutoff = 140 #@param {type:"number"}
#@markdown <i>*Units: donor–acceptor distance cutoff [Å], box side length [nm], donor–hydrogen–acceptor angle cutoff [°]<i>

add_water_bonds(u)
print(f"Bonds added: {len(u.bonds)}")

hbonds = HBA(universe=u,hydrogens_sel='name H1 H2',donors_sel='name O',acceptors_sel='name O',update_selections=False,d_a_cutoff=d_a_cutoff,d_h_a_angle_cutoff=d_h_a_angle_cutoff)
hbonds.run()

counts = hbonds.count_by_time()
n_waters = len(u.select_atoms("resname HOH and name O"))
hbonds_per_water = 2*counts/n_waters

plt.figure(figsize=(6,4))
title = f"Hydrogen bonds ({d_a_cutoff/10:.2f} nm, {d_h_a_angle_cutoff:d} deg)"
plt.plot(np.arange(hbonds_per_water.size)*2e-6*5000,hbonds_per_water)
plt.xlabel("Time (ns)")
plt.ylabel("Hydrogen bonds per water molecule")
plt.title(title)
plt.tight_layout()
plt.savefig(f'{system_name}/figures/{title.replace(' ','_')}.jpg',dpi=600)
plt.show()

print(f"Average hydrogen bonds per water molecule: {hbonds_per_water.mean():.2f}")

# 8. Step-by-step calculation of the radial distribution function

In [ ]:
# @title 8.1: Calculate histogram of oxygen–oxygen distances
#@markdown **Questions**
#@markdown
#@markdown 1. Why do the raw pair counts initially increase with distance?
#@markdown
#@markdown 2. How does changing the histogram bin width affect the smoothness and statistical noise of the distribution?
#@markdown
#@markdown 3. Why do the raw pair counts decrease again near the largest sampled distances?
#@markdown
#@markdown 4. For a cubic box of side length $L$, what is the largest distance for which a complete spherical shell can be sampled using periodic boundary conditions and the minimum-image convention?

bin_width = 0.1 #@param {type:"number"}
#@markdown <i>*Units: bin width [Å]<i>

from MDAnalysis.lib.distances import self_distance_array

oxygen = u.select_atoms("resname HOH and name O")
distances = []

for ts in u.trajectory:
    d = self_distance_array(oxygen.positions,box=ts.dimensions)
    distances.append(d)

distances = np.concatenate(distances)

print(f"Number of distances: {len(distances)}")
print(f"Minimum distance: {distances.min():.2f} Å")

edges = np.arange(0,d.max()+bin_width,bin_width)
counts,edges = np.histogram(distances,bins=edges)
r = 0.5*(edges[:-1]+edges[1:])

plt.figure(figsize=(6,4))
title = "Histogram of O–O distances"
plt.bar(r,counts,width=bin_width)
plt.xlabel(r"$r$ (Å)")
plt.ylabel("Pair counts")
plt.title(title)
plt.tight_layout()
plt.savefig(f'{system_name}/figures/{title.replace(' ','_')}.jpg',dpi=600)
plt.show()

In [ ]:
# @title 8.2: Calculate shell volumes

#@markdown The RDF counts neighbors inside spherical shells between $r_i$ and $r_{i+1}$.
#@markdown
#@markdown The exact volume of each shell is
#@markdown
#@markdown $$\Delta V_i=\frac{4\pi}{3}\left(r_{i+1}^3-r_i^3\right).$$
#@markdown
#@markdown Shells at larger distances have larger volumes, so they contain more particles even in a system of noninteracting particles with uniform bulk density. This geometric effect must be removed when normalizing the raw distance histogram.
#@markdown
#@markdown **Questions**
#@markdown
#@markdown 1. Why does the shell volume increase with distance even though the bin width is constant?
#@markdown
#@markdown 2. For narrow bins, the shell volume can be approximated as $\Delta V \approx 4\pi r^2\Delta r$. How does this explain the initial increase in the raw pair-count histogram?

shell_volumes = 4*np.pi/3*(edges[1:]**3-edges[:-1]**3)

plt.figure(figsize=(6,4))
title = "Volume of spherical shells"
plt.plot(r,shell_volumes)
plt.xlabel(r"$r$ (Å)")
plt.ylabel(r"Shell volume (Å$^3$)")
plt.title(title)
plt.tight_layout()
plt.savefig(f'{system_name}/figures/{title.replace(' ','_')}.jpg',dpi=600)
plt.show()

In [ ]:
# @title Step 4: Normalize the histogram

#@markdown In an NPT simulation, the box volume changes from frame to frame. We therefore normalize the distance histogram separately for every frame.
#@markdown
#@markdown For each frame:
#@markdown
#@markdown 1. We count how many unique O–O pairs fall inside each spherical shell.
#@markdown 2. We divide these counts by the shell volume, $\Delta V(r)$, to obtain the pair density at distance $r$.
#@markdown 3. We divide by the bulk pair density in the same frame:
#@markdown
#@markdown $$\rho_{\mathrm{bulk},t}=\frac{N_{\mathrm{pairs}}}{V_t},$$
#@markdown
#@markdown where $N_{\mathrm{pairs}}=N(N-1)/2$ and $V_t$ is the instantaneous box volume.
#@markdown
#@markdown The RDF for frame $t$ is therefore
#@markdown
#@markdown $$g_t(r)=\frac{N_t(r)/\Delta V(r)}{N_{\mathrm{pairs}}/V_t}.$$
#@markdown
#@markdown Finally, we average the framewise RDFs:
#@markdown
#@markdown $$g(r)=\left\langle g_t(r)\right\rangle_t.$$
#@markdown
#@markdown A value of $g(r)=1$ means that O–O pairs occur at distance $r$ with the same probability as in a system of noninteracting particles with the same bulk density.
#@markdown
#@markdown **Questions**
#@markdown
#@markdown 1. What does $g(r)>1$ tell us about the probability of finding two oxygen atoms separated by distance $r$?
#@markdown
#@markdown 2. Why is $g(r)$ close to zero at very short distances?
#@markdown
#@markdown 3. What structural feature of liquid water is represented by the first maximum of $g_{\mathrm{OO}}(r)$?
#@markdown
#@markdown 4. For a cubic box with side length $L$, what is the largest distance from a central particle for which a complete spherical shell can be sampled using the minimum-image convention?

n_pairs = len(oxygen)*(len(oxygen)-1)/2
g_frames = []

for ts in u.trajectory:
    distances = self_distance_array(oxygen.positions,box=ts.dimensions)
    frame_counts = np.histogram(distances,bins=edges)[0]

    volume = np.prod(ts.dimensions[:3])
    pair_density = frame_counts/shell_volumes
    bulk_pair_density = n_pairs/volume

    g_frames.append(pair_density/bulk_pair_density)

g_frames = np.asarray(g_frames)
g_r = g_frames.mean(axis=0)

plt.figure(figsize=(6,4))
title = "Oxygen–oxygen RDF"
plt.plot(r,g_r)
plt.xlabel(r"$r$ (Å)")
plt.ylabel(r"$g_{\mathrm{OO}}(r)$")
plt.title(title)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{system_name}/figures/{title.replace(' ','_')}.jpg',dpi=600)
plt.show()

In [ ]:
# @title 9. Download results

# @markdown In this zip file:

# @markdown `NPT_traj.dcd`: the NPT trajectory file;

# @markdown `EM_top.pdb`: the energy-minimized topology file;

# @markdown Plots generated during the analysis.

files_to_download = [f'{system_name}/NPT_traj.dcd',
                     f'{system_name}/EM_top.pdb',
                     f'{system_name}/figures']

import subprocess, glob
from google.colab import files

for filename in glob.glob(f'{system_name}/*'):
    if filename not in files_to_download:
        try:
            os.remove(f'{filename:s}')
        except:
            shutil.rmtree(f'{filename:s}')

zipper = f'zip -r {system_name}.zip {system_name}'
subprocess.run(zipper.split())
files.download(f'{system_name}.zip')